In [1]:
!pip install qiskit qiskit-aer qiskit-algorithms pandas scikit-learn matplotlib numpy scipy

In [ ]:

# GenoQ — Full Pipeline 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LassoCV
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from scipy.optimize import minimize

SEED = 42
np.random.seed(SEED)
simulator = AerSimulator()

# --- Load and clean training data ---
df_genes = pd.read_csv("C:/Users/HP/Downloads/archive/data_set_ALL_AML_train.csv")
cols_to_keep = ["Gene Accession Number"] + \
               [col for col in df_genes.columns
                if col not in ["Gene Description", "Gene Accession Number"]
                and "call" not in col.lower()]
df_clean = df_genes[cols_to_keep].set_index("Gene Accession Number")
df_transposed = df_clean.T

# --- Attach training labels ---
df_labels = pd.read_csv("C:/Users/HP/Downloads/archive/actual.csv")
df_labels['diagnosis'] = (df_labels['cancer'] == 'AML').astype(int)
df_labels = df_labels.set_index('patient')
df_transposed.index = df_transposed.index.astype(int)
df_final = df_transposed.copy()
df_final['diagnosis'] = df_labels['diagnosis']

# --- Prepare training data ---
X_real = df_final.drop(columns=["diagnosis"]).values
y_real = df_final["diagnosis"].values
gene_names_real = df_final.drop(columns=["diagnosis"]).columns.tolist()
scaler_real = MinMaxScaler()
X_real_scaled = scaler_real.fit_transform(X_real)

print("⏳ Computing MI scores — please wait...")
mi_scores_real = mutual_info_classif(X_real_scaled, y_real, random_state=SEED)

# --- Select top 4 genes ---
N_QUBITS_REAL = 4
top_indices_real = np.argsort(mi_scores_real)[::-1][:N_QUBITS_REAL]
top_genes_real = [gene_names_real[i] for i in top_indices_real]
top_mi_real = mi_scores_real[top_indices_real]

# --- Build QUBO matrix ---
Q_real = np.zeros((N_QUBITS_REAL, N_QUBITS_REAL))
for i in range(N_QUBITS_REAL):
    Q_real[i, i] = -top_mi_real[i]
    for j in range(i + 1, N_QUBITS_REAL):
        Q_real[i, j] = 0.1

# --- QAOA circuit builder ---
def build_qaoa_circuit(Q, gamma, beta, n_qubits):
    qc = QuantumCircuit(n_qubits)
    for q in range(n_qubits):
        qc.h(q)
    qc.barrier()
    for i in range(n_qubits):
        for j in range(i + 1, n_qubits):
            if abs(Q[i, j]) > 1e-8:
                qc.rzz(2 * gamma * Q[i, j], i, j)
        if abs(Q[i, i]) > 1e-8:
            qc.rz(2 * gamma * Q[i, i], i)
    qc.barrier()
    for q in range(n_qubits):
        qc.rx(2 * beta, q)
    qc.barrier()
    qc.measure_all()
    return qc

# --- QAOA optimization ---
cost_history_real = []
def compute_cost_real(params):
    gamma, beta = params
    qc = build_qaoa_circuit(Q_real, gamma, beta, N_QUBITS_REAL)
    compiled = transpile(qc, simulator)
    counts = simulator.run(compiled, shots=1024).result().get_counts()
    total_cost = 0.0
    total_shots = sum(counts.values())
    for bitstring, count in counts.items():
        x = np.array([int(b) for b in reversed(bitstring)])
        total_cost += (count / total_shots) * (x @ Q_real @ x)
    cost_history_real.append(total_cost)
    return total_cost

print("⏳ Running QAOA optimization — please wait 1-2 minutes...")
result_real = minimize(compute_cost_real, [0.5, 0.3],
                       method="COBYLA", options={"maxiter": 100})

# --- Extract selected genes ---
final_circuit = build_qaoa_circuit(Q_real, result_real.x[0],
                                   result_real.x[1], N_QUBITS_REAL)
compiled_final = transpile(final_circuit, simulator)
final_counts = simulator.run(compiled_final, shots=4096).result().get_counts()
best_bitstring = max(final_counts, key=final_counts.get)
best_x = np.array([int(b) for b in reversed(best_bitstring)])
selected_genes_real = [top_genes_real[i] for i, s in enumerate(best_x) if s == 1]

# --- Load and prepare test data ---
df_test = pd.read_csv("C:/Users/HP/Downloads/archive/data_set_ALL_AML_independent.csv")
cols_to_keep_test = ["Gene Accession Number"] + \
                    [col for col in df_test.columns
                     if col not in ["Gene Description", "Gene Accession Number"]
                     and "call" not in col.lower()]
df_test_clean = df_test[cols_to_keep_test].set_index("Gene Accession Number")
df_test_transposed = df_test_clean.T
df_test_transposed.index = df_test_transposed.index.astype(int)
df_test_final = df_test_transposed.copy()
df_test_final['diagnosis'] = df_labels['diagnosis']
X_test = df_test_final.drop(columns=["diagnosis"]).values
y_test = df_test_final["diagnosis"].values
X_test_scaled = scaler_real.transform(X_test)

# --- Train classifier on selected genes ---
selected_indices = [gene_names_real.index(g) for g in selected_genes_real]
X_train_q = X_real_scaled[:, selected_indices]
X_test_q  = X_test_scaled[:, selected_indices]
clf = RandomForestClassifier(n_estimators=100, random_state=SEED)
clf.fit(X_train_q, y_real)
acc_quantum = accuracy_score(y_test, clf.predict(X_test_q))

print("\n✅ Full pipeline complete!")
print(f"   Selected genes : {selected_genes_real}")
print(f"   Test accuracy  : {acc_quantum:.1%}")
print("\n🚀 Ready to build the Streamlit app!")

⏳ Computing MI scores — please wait...
⏳ Running QAOA optimization — please wait 1-2 minutes...

✅ Full pipeline complete!
   Selected genes : ['X95735_at', 'M55150_at', 'M27891_at', 'D10495_at']
   Test accuracy  : 91.2%

🚀 Ready to build the Streamlit app!
